# Phase 6A — Interactive PM Evidence Card

## 1. PM question

**Is the current momentum environment becoming fragile, what is driving the
change, and what evidence would confirm or invalidate the warning?**

This system *monitors fragility conditions*. It does **not** claim perfect
momentum-crash prediction. Momentum crashes are rare and state-dependent, so
the deterministic rules describe conditions under which a rebound could
squeeze the recent-loser leg — not a forecast that a crash will occur.

## 2. Architecture

```
Structured market data
        |
Deterministic risk indicators   (Phase 1 regime, Phase 3 leg risk)
        |
Risk-state and trigger engine   (Phase 4 four-row scorecard)
        |
Point-in-time evidence retrieval (cached, cutoff-enforced)
        |
Optional structured LLM synthesis (narrative text only; numbers untouched)
        |
PM Evidence Card
```

Every number on the card comes from the frozen Phase 1–4 code paths. The
optional synthesis layer only phrases narrative text and can never write a
quantitative field.

In [1]:
# Bootstrap: make `import src` work whether the kernel starts in the repo
# root or in notebooks/.
import sys, pathlib
_root = pathlib.Path.cwd()
if not (_root / 'src').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pandas as pd
from IPython.display import HTML, Markdown
from src.mvp.evidence_card import (
    build_evidence_card,
    render_evidence_card_html,
    render_signal_table_html,
)

In [2]:
# ============================================================
# PARAMETERS — EDIT THESE, THEN RUN ALL TO RECOMPUTE THE CARD
# ============================================================
AS_OF_DATE = "2024-01-05"        # worked example: quant history + cached evidence
COMPARE_TO_DATE = "2023-12-01"   # set to None to skip change analysis
THRESHOLD_PROFILE = "default"    # only 'default' is supported
USE_LLM = True                    # runs offline; deterministic narrative by default

# Other dates to try:
#   AS_OF_DATE = "2026-05-29"  -> current 'normal' state; evidence fails safe
#   AS_OF_DATE = "2020-03-24"  -> panic_elevated state (no cached evidence)

In [3]:
card = build_evidence_card(
    as_of_date=pd.Timestamp(AS_OF_DATE),
    compare_to_date=pd.Timestamp(COMPARE_TO_DATE) if COMPARE_TO_DATE else None,
    threshold_profile=THRESHOLD_PROFILE,
    use_llm=USE_LLM,
)
print('run_id      :', card.run_id)
print('risk state  :', card.overall_risk_state)
print('evidence    :', card.evidence_quality)
print('synthesis   :', card.synthesis_mode)

run_id      : 09cd5b545e9a8019
risk state  : bear_low_volatility
evidence    : available
synthesis   : deterministic_template


## 3. Inputs and run metadata

In [4]:
tail = 'unavailable' if card.tail_loss_frequency is None else f'{card.tail_loss_frequency:.2%}'
Markdown(
    '| Field | Value |\n|---|---|\n'
    f'| Selected date | `{card.as_of_date}` |\n'
    f'| Comparison date | `{card.comparison_date}` |\n'
    f'| Threshold profile | `{card.threshold_profile}` |\n'
    f'| Data cutoff | `{card.data_cutoff}` |\n'
    f'| LLM enabled | `{card.llm_enabled}` ({card.synthesis_mode}) |\n'
    f'| {card.tail_loss_horizon_days}-day conditional tail-loss freq (descriptive) | {tail} |\n'
    f'| Run ID | `{card.run_id}` |\n'
)

| Field | Value |
|---|---|
| Selected date | `2024-01-05` |
| Comparison date | `2023-12-01` |
| Threshold profile | `default` |
| Data cutoff | `2024-01-05T16:00:00-05:00` |
| LLM enabled | `True` (deterministic_template) |
| 20-day conditional tail-loss freq (descriptive) | 8.17% |
| Run ID | `09cd5b545e9a8019` |


## 4. Quantitative state

The four deterministic Phase 4 signals. `Value`, `Threshold`, `State`, and
`Δ vs compare` are computed only by the frozen quantitative pipeline.

In [5]:
HTML(render_signal_table_html(card))

Signal,Value,Threshold,State,Δ vs compare,Interpretation
high_volatility_recovery,0.0000,1.0000,not_triggered,+0.0000,"One composite macro gate replaces separate drawdown, recovery, and volatility alerts. It requires Phase 1 early recovery and high realized volatility to be true together."
short_minus_long_beta_gap,-0.5179,0.7116,not_triggered,-0.2747,Positive and unusually high short-underlying minus long beta indicates that a market rebound can squeeze the recent-loser leg. Threshold: prior-only 80th percentile from 1681 observations; raw historical threshold=0.71161.
portfolio_drawdown,-0.0903,-0.2000,not_triggered,+0.0098,"Long-short wealth relative to its highest level in the prior 63 trading days is compared with its own prior-only left-tail history. The threshold can never be looser than -20%, and drawdowns shallower than 5% are not material. Threshold: prior-only 20th percentile from 1681 observations; raw historical threshold=-0.20139; overridden by structural floor=-0.2; active threshold=-0.2."
short_loss_in_recovery,0.2039,0.2905,not_triggered,-0.0599,Trailing 21-day short loss magnitude is the sum of negative signed short contributions. It triggers only when Phase 1 early recovery is active and the loss reaches the threshold. Threshold: prior-only 80th percentile from 1723 observations; raw historical threshold=0.290521.


## 5. What changed

In [6]:
Markdown('\n'.join(f'- {line}' for line in card.what_changed))

- short_minus_long_beta_gap: -0.2433 -> -0.5179 (change -0.2747) versus 2023-12-01.
- short_loss_in_recovery: 0.2638 -> 0.2039 (change -0.0599) versus 2023-12-01.
- portfolio_drawdown: -0.1001 -> -0.0903 (change +0.0098) versus 2023-12-01.
- high_volatility_recovery: 0.0000 -> 0.0000 (change +0.0000) versus 2023-12-01.

## 6. Retrieved evidence

Point-in-time only: nothing published after the data cutoff is shown. The
evidence organizes context; it does not establish causality and cannot alter
the deterministic signals above.

In [7]:
def _ev(items):
    return '\n'.join(
        f'- **{e.headline_or_summary}** — {e.source} · {e.timestamp}'
        for e in items
    ) or '- _none retrieved at this cutoff_'

Markdown(
    '**Supporting**\n\n' + _ev(card.supporting_evidence)
    + '\n\n**Contradicting / moderating**\n\n'
    + _ev(card.contradicting_evidence + card.contextual_evidence)
    + '\n\n**Missing / uncertain**\n\n'
    + ('\n'.join(f'- {m}' for m in card.missing_or_uncertain_evidence) or '- _none_')
)

**Supporting**

- **Employment Situation, December 2023** — US Bureau of Labor Statistics · 2024-01-05T08:30:00-05:00
- **Personal Income and Outlays, November 2023** — US Bureau of Economic Analysis · 2023-12-22T08:30:00-05:00
- **Gross Domestic Product, Third Estimate, Third Quarter 2023** — US Bureau of Economic Analysis · 2023-12-21T08:30:00-05:00

**Contradicting / moderating**

- **Wall Street set for weak open after stronger jobs data** — Reuters via MarketScreener · 2024-01-05T09:08:00-05:00
- **Factors Affecting Reserve Balances, January 4, 2024** — Federal Reserve Board · 2024-01-04T16:30:00-05:00
- **Job Openings and Labor Turnover, November 2023** — US Bureau of Labor Statistics · 2024-01-03T10:00:00-05:00
- **FOMC statement, December 13, 2023** — Federal Reserve Board · 2023-12-13T14:00:00-05:00

**Missing / uncertain**

- The cached corpus is small and may omit relevant contradictory evidence.
- Generic macro context does not establish momentum-specific causality.
- Evidence cannot change deterministic metrics, thresholds, triggered states, or create a risk score.

## 7. Final Evidence Card

In [8]:
HTML(render_evidence_card_html(card))

Signal,Value,Threshold,State,Δ vs compare,Interpretation
high_volatility_recovery,0.0000,1.0000,not_triggered,+0.0000,"One composite macro gate replaces separate drawdown, recovery, and volatility alerts. It requires Phase 1 early recovery and high realized volatility to be true together."
short_minus_long_beta_gap,-0.5179,0.7116,not_triggered,-0.2747,Positive and unusually high short-underlying minus long beta indicates that a market rebound can squeeze the recent-loser leg. Threshold: prior-only 80th percentile from 1681 observations; raw historical threshold=0.71161.
portfolio_drawdown,-0.0903,-0.2000,not_triggered,+0.0098,"Long-short wealth relative to its highest level in the prior 63 trading days is compared with its own prior-only left-tail history. The threshold can never be looser than -20%, and drawdowns shallower than 5% are not material. Threshold: prior-only 20th percentile from 1681 observations; raw historical threshold=-0.20139; overridden by structural floor=-0.2; active threshold=-0.2."
short_loss_in_recovery,0.2039,0.2905,not_triggered,-0.0599,Trailing 21-day short loss magnitude is the sum of negative signed short contributions. It triggers only when Phase 1 early recovery is active and the loss reaches the threshold. Threshold: prior-only 80th percentile from 1723 observations; raw historical threshold=0.290521.


## 8. Historical context

State-conditional tail-loss frequencies from the matured-label history (`build_insurance_table`). These are descriptive base rates by regime state, not a claim that history must repeat.

In [9]:
pd.DataFrame(card.historical_analogs)

,state,horizon_days,sample_size,tail_loss_frequency,mean_forward_return,fifth_percentile_forward_return,latest_label_available_date,note
0,all,20,25029,0.051340,0.005122,-0.059398,2024-01-05,Descriptive state-conditional tail-loss freque...
1,normal,20,20538,0.034132,0.007314,-0.046874,2024-01-05,Descriptive state-conditional tail-loss freque...
2,bear_low_volatility,20,3110,0.081672,-0.001309,-0.079154,2024-01-04,Descriptive state-conditional tail-loss freque...
3,panic_elevated,20,1381,0.238957,-0.012991,-0.181668,2020-05-04,Descriptive state-conditional tail-loss freque...


## 9. Limitations

- Crowding and positioning proxies may not represent actual positions.
- Retrieval coverage is a small cached corpus and may be incomplete; missing
  evidence is uncertainty, not a benign finding.
- Narrative synthesis organizes evidence but does not prove causality.
- Thresholds are prior-only research rules, not optimized trading instructions.
- A sector selloff is **not** automatically a canonical UMD momentum crash.
- The synthetic portfolio uses a current-membership S&P 500 proxy and is
  survivorship-biased for historical dates.